# CARFAC vs mel spectrogram

Minimal demo: run the same short audio through two common frontends and look at them side by side.

| Frontend | What it is |
|----------|------------|
| **Mel spectrogram** | STFT → mel filterbank → dB (librosa default path) |
| **CARFAC NAP** | Cascade of Asymmetric Resonators with Fast-Acting Compression → neural activity pattern ([google/carfac](https://github.com/google/carfac) NumPy) |

Not a bake-off — just a visual sense of how the two representations differ on ordinary audio.

**Paid detection pilots** (hydrophone, drone, bird): [6cubed.app/#work](https://6cubed.app/#work) · [CARFAC pilots](https://github.com/6cubed/216labs/tree/main/docs/carfac-pilots) · [paid-pilot issue](https://github.com/6cubed/216labs/issues/new?template=paid-pilot.yml)

In [ ]:
# Install (Colab / fresh env). Safe to re-run.
%pip install -q "numpy" "matplotlib" "librosa" "soundfile" \
  "carfac @ git+https://github.com/google/carfac.git@master#subdirectory=python"

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display

from carfac.np import carfac as carfac_np

SR = 22050  # CARFAC NumPy default design rate
MAX_SECONDS = 4.0
RNG = np.random.default_rng(216)

## 1. Load audio

Default: a short public librosa example (randomly chosen from a small set).

Optional: set `UPLOAD_PATH` to a local/Colab-uploaded file path to use your own clip.

In [ ]:
UPLOAD_PATH = None  # e.g. "/content/my_clip.wav" after uploading in Colab

EXAMPLES = ("trumpet", "brahms", "nutcracker", "choice")


def load_clip(path: str | None = None) -> tuple[np.ndarray, int, str]:
    if path:
        y, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_SECONDS)
        return y.astype(np.float32), SR, path
    name = EXAMPLES[int(RNG.integers(0, len(EXAMPLES)))]
    y, _ = librosa.load(librosa.ex(name), sr=SR, mono=True, duration=MAX_SECONDS)
    return y.astype(np.float32), SR, f"librosa.ex({name!r})"


y, sr, source = load_clip(UPLOAD_PATH)
y = y / (np.max(np.abs(y)) + 1e-9)
print(f"source={source}  sr={sr}  samples={len(y)}  duration={len(y)/sr:.2f}s")
display(Audio(y, rate=sr))

## 2. Mel spectrogram

In [ ]:
n_mels = 96
n_fft = 1024
hop_length = 256

mel = librosa.feature.melspectrogram(
    y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, fmin=20.0, fmax=sr / 2
)
mel_db = librosa.power_to_db(mel, ref=np.max)
print("mel_db shape (n_mels, n_frames):", mel_db.shape)

## 3. CARFAC neural activity pattern (NAP)

`run_segment` returns NAP with shape `(n_samples, n_channels)`. We frame it (mean over hops) so the plot lines up roughly with the mel time axis.

In [ ]:
def carfac_nap(y_mono: np.ndarray, fs: int) -> tuple[np.ndarray, np.ndarray]:
    """Return (n_samples, n_ch) NAP and pole center frequencies (Hz)."""
    cfp = carfac_np.design_carfac(n_ears=1, fs=float(fs))
    cfp = carfac_np.carfac_init(cfp)
    # column vector: one ear
    waves = y_mono.astype(np.float64).reshape(-1, 1)
    naps, cfp, _bm, _ohc, _agc = carfac_np.run_segment(cfp, waves)
    # mono → (n_samp, n_ch)
    if naps.ndim == 3:
        naps = naps[:, :, 0]
    pole_freqs = np.asarray(cfp.pole_freqs, dtype=np.float64).reshape(-1)
    return naps.astype(np.float32), pole_freqs


def frame_mean(x: np.ndarray, frame: int, hop: int) -> np.ndarray:
    """x: (n_samp, n_ch) → (n_ch, n_frames) mean absolute activity per hop."""
    n_samp, n_ch = x.shape
    frames = []
    for start in range(0, max(n_samp - frame + 1, 1), hop):
        chunk = x[start : start + frame]
        if chunk.shape[0] < frame:
            break
        frames.append(np.mean(np.abs(chunk), axis=0))
    if not frames:
        frames = [np.mean(np.abs(x), axis=0)]
    return np.stack(frames, axis=1)  # (n_ch, n_frames)


naps, pole_freqs = carfac_nap(y, sr)
nap_framed = frame_mean(naps, frame=n_fft, hop=hop_length)
nap_db = 20.0 * np.log10(nap_framed + 1e-8)
nap_db = nap_db - np.max(nap_db)
print("naps shape (n_samp, n_ch):", naps.shape)
print("nap_db shape (n_ch, n_frames):", nap_db.shape)
print(f"CARFAC channels: {len(pole_freqs)}  CF range: {pole_freqs.min():.1f}–{pole_freqs.max():.1f} Hz")

## 4. Side-by-side look

In [ ]:
fig = plt.figure(figsize=(12, 8), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 2.2, 2.2])

ax_w = fig.add_subplot(gs[0])
t = np.arange(len(y)) / sr
ax_w.plot(t, y, color="#222", lw=0.6)
ax_w.set_xlim(0, t[-1] if len(t) else 1)
ax_w.set_ylabel("amp")
ax_w.set_title(f"Waveform — {source}")
ax_w.set_xlabel("time (s)")

ax_m = fig.add_subplot(gs[1])
img_m = librosa.display.specshow(
    mel_db,
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="mel",
    fmin=20.0,
    fmax=sr / 2,
    ax=ax_m,
    cmap="magma",
)
ax_m.set_title(f"Mel spectrogram ({n_mels} mels, n_fft={n_fft}, hop={hop_length})")
fig.colorbar(img_m, ax=ax_m, format="%+2.0f dB")

ax_c = fig.add_subplot(gs[2])
# Channels are ordered high CF → low CF in CARFAC; flip so low freq is at bottom (like mel).
nap_plot = nap_db[::-1]
extent = [0, len(y) / sr, float(pole_freqs.min()), float(pole_freqs.max())]
img_c = ax_c.imshow(
    nap_plot,
    aspect="auto",
    origin="lower",
    extent=extent,
    cmap="magma",
    interpolation="nearest",
)
ax_c.set_ylabel("approx CF (Hz)")
ax_c.set_xlabel("time (s)")
ax_c.set_title(f"CARFAC NAP (framed |mean|, {len(pole_freqs)} channels) — dB re peak")
fig.colorbar(img_c, ax=ax_c, format="%+2.0f dB")

plt.show()

print(
    "Read tip: mel is linear filterbank energy in STFT frames; "
    "CARFAC NAP is a nonlinear cochlear model (compression + AGC) so onsets and "
    "level dynamics often look different even on the same clip."
)

## Optional: re-roll another random example

Re-run the load cell (and cells below) to draw a different `librosa.ex(...)` sample, or set `UPLOAD_PATH`.